# デートBot Phase A（Colab）

要件: 複数ターンで条件ヒアリング → 自前LoRA（口調・好み）→ Gemini（案3つ＋理由）

**事前準備**
1. ランタイムを GPU に変更
2. 左の鍵アイコン Secrets に `GEMINI_API_KEY` を登録（Google AI Studio の無料キー）
3. `date_bot_train.csv` をアップロード（後のセル）
4. 任意で `date_bot_preferences_summary.md` もアップロード

店舗CSVは使いません。エリア＋好み要約のみです。


## 0. ライブラリインストール


In [12]:
print('transformers をバージョン 4.38.2 にダウングレードします。これまでの環境が壊れる可能性があるため、注意して実行してください。')
!pip install -U "transformers==4.38.2"

transformers をバージョン 4.38.2 にダウングレードします。これまでの環境が壊れる可能性があるため、注意して実行してください。
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.14.1
    Uninstalling transformers-5.14.1:
      Successfully uninstalled transformers-5.14.1
ERROR: pip's dependency resolver does not currently take into account

In [3]:
# Colab互換: pandasは上げない（google-colab/cudfと衝突するため）
# peft新版は torchao>=0.16 が必要
!pip -q install -U "transformers>=4.45" "datasets" "accelerate" "peft" "google-generativeai"
!pip -q install -U "torchao>=0.16.0"
# 念のため Colab 標準付近に戻す（既に壊れている場合）
!pip -q install "pandas==2.2.2"



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.7 MB/s eta 0:00:00


## 1. 設定・好み要約の読み込み


In [4]:
import os, re, json, torch
from pathlib import Path

USE_GPU = torch.cuda.is_available()
print("GPU available:", USE_GPU)

# 軽量かつ日本語指示が比較的強いモデル（OOMなら 0.5B に変更）
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # VRAM不足時

DEFAULT_BUDGET = 3000
PRIORITY_AREAS = ["渋谷", "上野", "新宿", "お台場", "品川"]

PREFERENCES_FALLBACK = (
    "呼称: リョウスケ君 / こゆたん\n"
    "エリア: 東京23区。優先は渋谷・上野・新宿／お台場〜品川\n"
    "予算デフォルト: 一人3000円\n"
    "好き: ご飯(おいしい+安い)、カフェ、映画、動物園・水族館、季節イベント、散歩・公園、スイーツ、猫カフェ、美術館・博物館\n"
    "NG: 高いのに微妙な店、長時間ダラダラ\n"
    "定番: イベント+ご飯、上野/渋谷起点、コスパ重視、微妙店のリベンジ\n"
    "口調: フランク。予約代行・在庫確認・恋愛深掘りはしない。個人情報は扱わない。"
)

prefs_path = Path("date_bot_preferences_summary.md")
if prefs_path.exists():
    PREFERENCES_TEXT = prefs_path.read_text(encoding="utf-8")
    print("Loaded preferences from file")
else:
    PREFERENCES_TEXT = PREFERENCES_FALLBACK
    print("Using embedded preferences fallback")

print(PREFERENCES_TEXT[:400], "...")



GPU available: True
Using embedded preferences fallback
呼称: リョウスケ君 / こゆたん
エリア: 東京23区。優先は渋谷・上野・新宿／お台場〜品川
予算デフォルト: 一人3000円
好き: ご飯(おいしい+安い)、カフェ、映画、動物園・水族館、季節イベント、散歩・公園、スイーツ、猫カフェ、美術館・博物館
NG: 高いのに微妙な店、長時間ダラダラ
定番: イベント+ご飯、上野/渋谷起点、コスパ重視、微妙店のリベンジ
口調: フランク。予約代行・在庫確認・恋愛深掘りはしない。個人情報は扱わない。 ...


## 2. モデル・Tokenizer・LoRA


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model

set_seed(42)
R = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16 if USE_GPU else torch.float32,
    device_map="auto" if USE_GPU else None,
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=R,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


## 3. 学習CSVの読み込み


In [6]:
import pandas as pd
from datasets import Dataset

STOP_TEXT = " ### END"
PROMPT_TMPL = "Input: {input}\n---\nOutput:"

csv_name = "date_bot_train_slots.csv"  # スロット抽出用データ
if not Path(csv_name).exists():
    try:
        from google.colab import files
        print("date_bot_train_slots.csv を選択してアップロード")
        uploaded = files.upload()
        csv_name = next(iter(uploaded.keys()))
    except Exception as e:
        raise FileNotFoundError("date_bot_train.csv をノートと同じ場所に置いてください") from e

df = pd.read_csv(csv_name)
assert "input" in df.columns and "output" in df.columns

def build_text(row):
    prompt = PROMPT_TMPL.format(input=str(row["input"]).strip())
    return prompt + " " + str(row["output"]).strip() + STOP_TEXT

df["text"] = df.apply(build_text, axis=1)
dataset = Dataset.from_pandas(df[["text"]])
print("Samples:", len(dataset))
print(dataset[0])


date_bot_train_slots.csv を選択してアップロード


Saving date_bot_train_slots.csv to date_bot_train_slots.csv
Samples: 80
{'text': 'Input: こゆたんと土曜の午後、上野で安く遊びたい\n---\nOutput: budget:3000\\ntime_slot:昼/午後\\narea:上野\\nmood:コスパ\\navoid_areas: ### END'}


## 4. トークナイズ（回答部分のみ学習）


In [7]:
MAX_LENGTH = 512

def tokenize_fn(examples):
    all_input_ids, all_attention_mask, all_labels = [], [], []
    for text in examples["text"]:
        encoded = tokenizer(text, truncation=True, max_length=MAX_LENGTH)
        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]

        prompt_text = text.split("Output:")[0] + "Output:"
        prompt_len = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])

        labels = input_ids.copy()
        for i in range(min(prompt_len, len(labels))):
            labels[i] = -100

        pad_len = MAX_LENGTH - len(input_ids)
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = attention_mask + [0] * pad_len
        labels = labels + [-100] * pad_len

        all_input_ids.append(input_ids)
        all_attention_mask.append(attention_mask)
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    }

tokenized_data = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
from transformers import default_data_collator
collator = default_data_collator
print("Train:", len(tokenized_data))



Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Train: 80


## 5. LoRA学習


In [8]:
from transformers import TrainingArguments, Trainer

EPOCHS = 15
LR = 2e-4
BATCH = 2

args = TrainingArguments(
    output_dir="./date_bot_lora",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    learning_rate=LR,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    bf16=USE_GPU,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data,
    data_collator=collator,
)
trainer.train()
print("done")


Step,Training Loss
5,4.260178
10,3.517469
15,2.827492
20,1.548367
25,1.318054
30,0.896974
35,0.478570
40,0.457267
45,0.323921
50,0.171055


KeyboardInterrupt: 

## 6. Gemini セットアップ（無料枠）


In [9]:
import os
import time
import requests

def get_gemini_api_key():
    if "GEMINI_API_KEY" in globals() and globals().get("GEMINI_API_KEY"):
        print("APIキー: 既存変数から取得")
        return str(globals()["GEMINI_API_KEY"]).strip()
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            print("APIキー: Colab Secrets から取得")
            return key.strip()
    except Exception as e:
        print(f"Colab Secrets 読み込み失敗: {e}")
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        print("APIキー: 環境変数から取得")
        return key.strip()
    return None

GEMINI_API_KEY = get_gemini_api_key()
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY がありません。")

# 確認済みモデル
GEMINI_MODEL_NAME = "gemini-3.1-flash-lite"


def gemini_rest_generate(prompt: str, model_name: str = None, timeout: int = 45) -> str:
    model_name = model_name or GEMINI_MODEL_NAME
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent"
    r = requests.post(
        url,
        params={"key": GEMINI_API_KEY},
        json={"contents": [{"parts": [{"text": prompt}]}]},
        timeout=timeout,
    )
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
    data = r.json()
    return data["candidates"][0]["content"]["parts"][0]["text"].strip()


sample = gemini_rest_generate("1語だけ返して", timeout=25)
print("Gemini REST ready:", GEMINI_MODEL_NAME, "|", sample)
gemini_model = GEMINI_MODEL_NAME

APIキー: Colab Secrets から取得
Gemini REST ready: gemini-3.1-flash-lite | 承知。


In [ ]:
# 診断用（任意）。絶対に変数名 model を使わないこと！
import requests
key = GEMINI_API_KEY
for cand_name in ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.5-flash"]:
    r = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{cand_name}:generateContent",
        params={"key": key},
        json={"contents": [{"parts": [{"text": "こんにちは。1語だけ返して。"}]}]},
        timeout=25,
    )
    print(cand_name, r.status_code, r.text[:150].replace("\n", " "))


## 7. 対話エンジン（LoRA=スロット抽出 / Gemini=提案）

1. LoRAが `budget/time_slot/area/mood/avoid_areas` をキーワード形式で出力（方針文なし）
2. slots にマージ
3. 不足なら定型で聞き返し
4. Gemini に **slots + 好み要約のみ** を渡し、案3つ＋理由


In [14]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
import re

GEMINI_TIMEOUT_SEC = 45

if isinstance(model, str):
    raise TypeError(
        "変数 model が文字列です。セクション2を再実行してください。"
        f" 現在: {model!r}"
    )
local_model = model
local_device = next(local_model.parameters()).device
print("local_model OK:", type(local_model).__name__, local_device)

SLOT_KEYS = ["budget", "time_slot", "area", "mood", "avoid_areas"]


@torch.inference_mode()
def local_generate(user_text: str, max_new_tokens: int = 120) -> str:
    """スロット抽出専用。方針文は出させない。"""
    messages = [
        {
            "role": "system",
            "content": (
                "あなたはデート条件の抽出器。"
                "次の形式だけを出力し、他の文章は禁止。"
                "budget:<円の数字 or 空>\n"
                "time_slot:<昼/午後|夜/夕方|朝から夜（終日）|空>\n"
                "area:<渋谷|上野|新宿|お台場|品川|池袋|空>\n"
                "mood:<キーワードを読点区切り or 空>\n"
                "avoid_areas:<除外エリアを読点区切り or 空>\n"
                "説明文・方針・提案は書かない。"
            ),
        },
        {"role": "user", "content": user_text},
    ]
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        prompt = PROMPT_TMPL.format(input=user_text)

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(local_device) for k, v in inputs.items()}
    out = local_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.05,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return text.split(STOP_TEXT)[0].strip()


def parse_slot_text(text: str) -> dict:
    """LoRA出力をdictへ。リテラル\\n（バックスラッシュ+n）にも対応。"""
    parsed = {k: None for k in SLOT_KEYS}
    if not text:
        return parsed

    # "budget:3000\ntime_slot:昼/午後" のような文字\\nを実改行へ
    if "\\n" in text and text.count("\n") <= 1:
        text = text.replace("\\n", "\n")
    text = text.replace("\\t", "\t").replace("\r", "")

    for line in text.split("\n"):
        line = line.strip().strip("`")
        if not line or ":" not in line:
            continue
        key, val = line.split(":", 1)
        key = key.strip().lstrip("-* ").strip()
        val = val.strip().strip('"').strip("'")
        if key not in parsed:
            continue
        if val in ("", "空", "なし", "None", "null", "-", "未定"):
            parsed[key] = None
        else:
            parsed[key] = val

    if parsed.get("budget") is not None:
        m = re.search(r"\d+", str(parsed["budget"]).replace(",", ""))
        parsed["budget"] = int(m.group()) if m else None

    if parsed.get("avoid_areas"):
        parts = re.split(r"[、,/\s]+", str(parsed["avoid_areas"]))
        parsed["avoid_areas"] = [p for p in parts if p and p not in ("空", "なし")]
    else:
        parsed["avoid_areas"] = []
    return parsed



def postprocess_slots(user_text: str, extracted: dict) -> dict:
    """LoRA抽出の後処理（幻覚抑制・部分更新）。"""
    t = str(user_text)
    # 全角数字を半角に
    t_norm = t.translate(str.maketrans("０１２３４５６７８９", "0123456789"))

    out = dict(extracted)
    out["avoid_areas"] = list(out.get("avoid_areas") or [])

    # --- avoid: 除外表現がないのに付いていたら捨てる ---
    neg_cues = ["以外", "やめ", "じゃなく", "じゃない", "除外", "なしで", "NG", "避け"]
    if not any(c in t for c in neg_cues):
        out["avoid_areas"] = []

    # --- time_slot: 時間語がないのに付いていたら捨てる ---
    time_cues = ["朝", "昼", "午後", "夜", "夕方", "ランチ", "ディナー", "終日", "一日", "晩"]
    if out.get("time_slot") and not any(c in t for c in time_cues):
        out["time_slot"] = None

    # --- area: エリア語がないのに付いていたら捨てる ---
    area_cues = ["渋谷", "上野", "新宿", "お台場", "品川", "池袋", "23区", "都内"]
    if out.get("area") and not any(c in t for c in area_cues):
        out["area"] = None

    # --- budget: ルールで補完（やっぱ5000 / 1万 など） ---
    m = re.search(r"(\d{3,6})\s*円", t_norm)
    if m:
        out["budget"] = int(m.group(1))
    m = re.search(r"(\d+(?:\.\d+)?)\s*万\s*円?", t_norm)
    if m:
        out["budget"] = int(float(m.group(1)) * 10000)
    m = re.search(r"(?:やっぱ|やはり|やっぱり|にする|で)\s*(\d{3,6})\b", t_norm)
    if m and out.get("budget") is None:
        out["budget"] = int(m.group(1))
    m = re.search(r"\b(\d{3,6})\b", t_norm)
    # 「やっぱ5000」だけ等、短文で数字1つのとき
    if out.get("budget") is None and re.fullmatch(r"\s*(?:やっぱ|やはり|やっぱり)?\s*\d{3,6}\s*円?\s*", t_norm):
        m = re.search(r"(\d{3,6})", t_norm)
        if m:
            out["budget"] = int(m.group(1))
    if any(k in t for k in ["安め", "安く", "コスパ", "抑えめ"]) and out.get("budget") is None:
        out["budget"] = 3000

    # --- 部分更新発話: 予算だけのときは他を触らない（抽出を空に）---
    budget_only = bool(re.fullmatch(
        r"\s*(?:予算は?|一人)?\s*(?:やっぱ|やはり|やっぱり)?\s*(?:\d+(?:\.\d+)?\s*万\s*円?|\d{3,6}\s*円?)\s*(?:で|にする|くらい)?\s*",
        t_norm,
    )) or bool(re.fullmatch(r"\s*(?:やっぱ|やはり|やっぱり)\s*\d{3,6}\s*円?\s*", t_norm))
    if budget_only:
        out["time_slot"] = None
        out["area"] = None
        out["mood"] = None
        # avoid は文に除外がなければ既に空
        if not any(c in t for c in neg_cues):
            out["avoid_areas"] = []

    # --- mood: キーワードが文にあれば補完 ---
    mood_map = {
        "カフェ": "カフェ", "映画": "映画", "動物園": "動物園", "水族館": "水族館",
        "ご飯": "ご飯", "食事": "ご飯", "イベント": "イベント", "散歩": "散歩",
        "美術館": "美術館", "博物館": "博物館", "スイーツ": "スイーツ",
        "猫カフェ": "猫カフェ", "コスパ": "コスパ", "安く": "コスパ", "安め": "コスパ",
    }
    moods = []
    if out.get("mood"):
        moods.extend(re.split(r"[、,/]", str(out["mood"])))
    for k, v in mood_map.items():
        if k in t and v not in moods:
            moods.append(v)
    moods = [m for m in moods if m and m not in ("空", "なし")]
    out["mood"] = "、".join(dict.fromkeys(moods)) if moods else None

    # --- avoid: 文からルール抽出も併用 ---
    areas = ["渋谷", "上野", "新宿", "お台場", "品川", "池袋"]
    avoided = list(out.get("avoid_areas") or [])
    for a in areas:
        if any(p in t for p in [f"{a}以外", f"{a}はやめ", f"{a}じゃなく", f"{a}じゃない", f"{a}は避け", f"{a}NG", f"{a}はNG"]):
            if a not in avoided:
                avoided.append(a)
    out["avoid_areas"] = avoided

    # 希望areaがavoidに入っていたらareaを消す
    if out.get("area") in (out.get("avoid_areas") or []):
        out["area"] = None

    return out


def merge_slots(slots: dict, extracted: dict) -> dict:
    """今回取れた値だけ上書き。avoidは累積。"""
    for k in ["budget", "time_slot", "area", "mood"]:
        v = extracted.get(k)
        if v is not None and v != "":
            slots[k] = v
    avoided = slots.get("avoid_areas", [])
    for a in extracted.get("avoid_areas") or []:
        if a not in avoided:
            avoided.append(a)
    if avoided:
        slots["avoid_areas"] = avoided
        # 除外と希望が衝突したら希望を消す
        if slots.get("area") in avoided:
            slots.pop("area", None)
    return slots


def missing_slots(slots: dict):
    need = []
    if "time_slot" not in slots:
        need.append("時間帯（昼/午後/夜/終日など）")
    if "area" not in slots:
        need.append("エリア（渋谷・上野・新宿／お台場〜品川など）")
    return need


def template_clarify(need, slots):
    msg = "おけ。もうちょい条件ほしい。"
    if need:
        msg += " " + " / ".join(need) + " を教えて。"
    avoid = slots.get("avoid_areas")
    if avoid:
        msg += f" （除外中: {'、'.join(avoid)}）"
    msg += " 例:『上野で午後、一人5000円』"
    return msg


def fallback_plans(slots: dict) -> str:
    area = slots.get("area", "渋谷")
    budget = slots.get("budget", DEFAULT_BUDGET)
    time_slot = slots.get("time_slot", "昼/午後")
    mood = slots.get("mood", "コスパ重視")
    templates = {
        "上野": [
            (f"上野動物園 → 上野公園短散策 → 安めご飯（一人{budget}円以内）", "定番・コスパ"),
            (f"博物館（常設寄り）→ カフェ", "屋内・午後向き"),
            (f"公園散歩 → ラーメン/定食 → 早め解散", "ダラダラ回避"),
        ],
        "渋谷": [
            (f"公園短散策 → カフェ → 安めご飯", "集合しやすい"),
            (f"映画 → 前後でファミレス/ラーメン", "予算管理しやすい"),
            (f"スイーツカフェ → 短散歩", "早め解散可"),
        ],
        "新宿": [
            (f"新宿御苑 → ランチ", "緑＋コスパ"),
            (f"映画 → 安めご飯", "屋内中心"),
            (f"カフェ → 短散策", "落ち着き"),
        ],
        "お台場": [
            (f"海浜公園散歩 → 食事", "景色＋安さ"),
            (f"無料スポット中心 → チェーン食事", "高額回避"),
            (f"午後のんびり → 早め切り上げ", "終了明確"),
        ],
        "品川": [
            (f"水族館（料金要確認）→ 抑えめ食事", "予算調整"),
            (f"短散歩 → 安めご飯", "移動少なめ"),
            (f"屋内 → カフェ", "天候に強い"),
        ],
    }
    key = area if area in templates else "渋谷"
    lines = [f"（簡易提案 / {area} / {time_slot} / {mood} / {budget}円）", ""]
    for i, (plan, reason) in enumerate(templates[key], 1):
        lines += [f"案{i}: {plan}", f"理由: {reason}", ""]
    return "\n".join(lines).strip()


def prefs_for_gemini() -> str:
    return (
        "呼称: リョウスケ君/こゆたん。東京23区。"
        "優先エリア: 渋谷・上野・新宿/お台場〜品川。"
        "好き: 安いおいしいご飯、カフェ、映画、動物園/水族館、季節イベント、散歩、スイーツ。"
        "NG: 高いのに微妙、長時間ダラダラ。"
        "定番: イベント+ご飯、上野/渋谷起点、コスパ。"
    )


def _gemini_call(prompt: str) -> str:
    return gemini_rest_generate(prompt, GEMINI_MODEL_NAME, timeout=GEMINI_TIMEOUT_SEC)


def gemini_plans(slots: dict) -> str:
    """方針文は渡さない。slots + 好み要約のみ。"""
    if not GEMINI_MODEL_NAME:
        print("Gemini未設定 → フォールバック")
        return fallback_plans(slots)

    budget = slots.get("budget", DEFAULT_BUDGET)
    prompt = (
        "デートプラン補助。予約代行・在庫確認・恋愛深掘り禁止。個人情報禁止。\n"
        f"制約: 東京23区、学生向けも意識。一人あたり目安{budget}円。超過なら理由を書く。\n"
        f"好み要約: {prefs_for_gemini()}\n"
        f"抽出済み条件slots: {json.dumps(slots, ensure_ascii=False)}\n"
        "avoid_areas は絶対に提案しない。\n"
        "フランクな日本語で厳守:\n"
        "案1: ...\n理由: ...\n案2: ...\n理由: ...\n案3: ...\n理由: ...\n"
        "実在名はできるだけ。不確実なら「候補:」。"
    )

    print(f"Gemini呼び出し中…（{GEMINI_MODEL_NAME} / 最大{GEMINI_TIMEOUT_SEC}秒）")
    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(_gemini_call, prompt)
            return fut.result(timeout=GEMINI_TIMEOUT_SEC + 5)
    except FuturesTimeout:
        print("Geminiタイムアウト → フォールバック")
        return fallback_plans(slots)
    except Exception as e:
        print(f"Gemini失敗 → フォールバック: {e}")
        return fallback_plans(slots)


print("engine ready (LoRA=slot抽出 / Gemini=提案のみ)")


local_model OK: PeftModelForCausalLM cuda:0
engine ready (LoRA=slot抽出 / Gemini=提案のみ)


## 8. 対話テスト（exit で終了）


In [16]:
slots = {}
print("デートBot起動（LoRAでスロット抽出 → Geminiで案3つ）")
print("例: 『こゆたんと土曜の午後、上野で安く遊びたい』 / reset / exit")

while True:
    user_input = input("\nあなた: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "exit":
        break
    if user_input.lower() == "reset":
        slots = {}
        print("Bot: 条件リセットしたよ。")
        continue

    if any(k in user_input for k in ["予約して", "空席", "恋愛相談", "復縁", "住所", "電話番号"]):
        print("Bot: それは対応外だよ。デート案の相談だけ続けるね。")
        continue

    if len(user_input) <= 1:
        print("Bot:", template_clarify(["時間帯", "エリア"], slots))
        print("(現在の条件)", slots)
        continue

    print("スロット抽出中（LoRA）…")
    raw = local_generate(user_input, max_new_tokens=96)
    print("LoRA生出力:\n", raw)
    extracted = parse_slot_text(raw)
    print("抽出結果(raw parse):", extracted)
    extracted = postprocess_slots(user_input, extracted)
    print("抽出結果(後処理後):", extracted)
    slots = merge_slots(slots, extracted)
    print("(マージ後slots)", slots)

    need = missing_slots(slots)
    if need:
        print("Bot:", template_clarify(need, slots))
        continue

    slots.setdefault("budget", DEFAULT_BUDGET)
    plans = gemini_plans(slots)  # 方針文なし
    print("Bot（提案）:\n" + plans)
    print("(現在の条件)", slots)


デートBot起動（LoRAでスロット抽出 → Geminiで案3つ）
例: 『こゆたんと土曜の午後、上野で安く遊びたい』 / reset / exit

あなた: 上野
スロット抽出中（LoRA）…
LoRA生出力:
 time_slot:昼/午後
area:上野
mood:
avoid_areas:渋谷、池袋、お台場、品川
 budget:（空）
end
抽出結果(raw parse): {'budget': None, 'time_slot': '昼/午後', 'area': '上野', 'mood': None, 'avoid_areas': ['渋谷', '池袋', 'お台場', '品川']}
抽出結果(後処理後): {'budget': None, 'time_slot': None, 'area': '上野', 'mood': None, 'avoid_areas': []}
(マージ後slots) {'area': '上野'}
Bot: おけ。もうちょい条件ほしい。 時間帯（昼/午後/夜/終日など） を教えて。 例:『上野で午後、一人5000円』

あなた: 朝から
スロット抽出中（LoRA）…
LoRA生出力:
 time_slot:朝から夜（終日）
area:渋谷、上野、新宿、お台場、品川、池袋
mood:空
avoid_areas: ### (無回避エリア)
抽出結果(raw parse): {'budget': None, 'time_slot': '朝から夜（終日）', 'area': '渋谷、上野、新宿、お台場、品川、池袋', 'mood': None, 'avoid_areas': ['###', '(無回避エリア)']}
抽出結果(後処理後): {'budget': None, 'time_slot': '朝から夜（終日）', 'area': None, 'mood': None, 'avoid_areas': []}
(マージ後slots) {'area': '上野', 'time_slot': '朝から夜（終日）'}
Gemini呼び出し中…（gemini-3.1-flash-lite / 最大45秒）
Bot（提案）:
リョウスケ君、こゆたん、お疲れ様！上野周辺で、コスパ重視＆ダラダラしないメリハリあるプランを3つ考えたよ。

-

KeyboardInterrupt: Interrupted by user

In [17]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/date_bot_lora"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("saved:", SAVE_DIR)

Mounted at /content/drive
saved: /content/drive/MyDrive/date_bot_lora


## 9. 追加学習の手順（運用）

1. `date_bot_train.csv` に input,output 行を追加
2. セクション3〜5を再実行
3. セクション8で対話確認

GitHub公開時は `date_bot_train_sample.csv` のみ載せる。APIキー・本番CSV・モデル本体は載せない。
